# 05 - Grad-CAM Interpretability

This notebook loads the saved transfer-learning classifier and explains correct and incorrect predictions with Grad-CAM.

## Google Colab Git Setup

Run the next cell only when using Google Colab. Set `REPO_URL` to your GitHub repository URL, then the cell clones or pulls the repository into `/content/PetVision-DeepLearning` and installs dependencies. If you run locally, skip it.


In [ ]:
# Colab-only Git setup. Skip this cell when running locally.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/YOUR_USERNAME/PetVision-DeepLearning.git"
PROJECT_DIR = Path('/content/PetVision-DeepLearning')

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    if 'YOUR_USERNAME' in REPO_URL:
        raise ValueError('Replace REPO_URL with your GitHub repository URL before running this cell.')
    if PROJECT_DIR.exists():
        subprocess.check_call(['git', '-C', str(PROJECT_DIR), 'pull'])
    else:
        subprocess.check_call(['git', 'clone', REPO_URL, str(PROJECT_DIR)])
    os.chdir(PROJECT_DIR)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
    print('Colab project root:', os.getcwd())
else:
    print('Not running in Google Colab. Continue with the local setup cells below.')


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

for path in [
    "models",
    "results/classification",
    "results/segmentation",
    "results/gradcam",
    "results/figures",
]:
    (PROJECT_ROOT / path).mkdir(parents=True, exist_ok=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from src.data_loader import get_splits, load_label_mapping
from src.preprocessing import preprocess_transfer_classification
from src.gradcam import make_nested_backbone_gradcam_heatmap, top_k_predictions
from src.visualization import overlay_heatmap

In [ ]:
MODEL_PATH = PROJECT_ROOT / "models/best_classifier.keras"
assert MODEL_PATH.exists(), "Run notebook 03 first to create models/best_classifier.keras"

classifier = tf.keras.models.load_model(MODEL_PATH)
label_mapping = load_label_mapping(PROJECT_ROOT / "results/figures/label_mapping.json")
label_names = [label_mapping[idx] for idx in sorted(label_mapping)]

_, _, test_raw, _, _ = get_splits()
prep = lambda example: preprocess_transfer_classification(example, backbone="mobilenet_v2")
test_ds = test_raw.map(prep, num_parallel_calls=tf.data.AUTOTUNE).batch(32).prefetch(tf.data.AUTOTUNE)

In [ ]:
# Collect a small pool of correct and incorrect predictions.
examples = []
for images, labels in test_ds.take(10):
    probs = classifier.predict(images, verbose=0)
    preds = np.argmax(probs, axis=1)
    for image, label, pred, prob in zip(images.numpy(), labels.numpy(), preds, probs):
        examples.append({"image": image, "label": int(label), "pred": int(pred), "prob": prob})

correct = [ex for ex in examples if ex["label"] == ex["pred"]][:4]
incorrect = [ex for ex in examples if ex["label"] != ex["pred"]][:4]
print(f"Correct examples: {len(correct)}")
print(f"Incorrect examples: {len(incorrect)}")

In [ ]:
def denormalize_mobilenet(image):
    return np.clip((image + 1.0) / 2.0, 0, 1)

def save_gradcam_grid(selected, output_path, title):
    if not selected:
        print(f"No examples available for {title}")
        return
    fig, axes = plt.subplots(len(selected), 2, figsize=(8, 4 * len(selected)))
    if len(selected) == 1:
        axes = np.array([axes])
    for row, ex in enumerate(selected):
        image_batch = ex["image"][np.newaxis, ...]
        heatmap = make_nested_backbone_gradcam_heatmap(
            image_batch,
            classifier,
            backbone_layer_name="mobilenetv2_1.00_224",
            last_conv_layer_name="Conv_1",
            pred_index=ex["pred"],
        )
        heatmap = tf.image.resize(heatmap[..., np.newaxis], (224, 224)).numpy().squeeze()
        image = denormalize_mobilenet(ex["image"])
        overlay = overlay_heatmap(image, heatmap)
        axes[row, 0].imshow(image)
        axes[row, 0].set_title(f"True: {label_names[ex['label']]}\nPred: {label_names[ex['pred']]}")
        axes[row, 1].imshow(overlay)
        axes[row, 1].set_title("Grad-CAM overlay")
        for col in range(2):
            axes[row, col].axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(output_path, dpi=160)
    plt.show()

save_gradcam_grid(correct, PROJECT_ROOT / "results/gradcam/correct_predictions_gradcam.png", "Correct predictions")
save_gradcam_grid(incorrect, PROJECT_ROOT / "results/gradcam/incorrect_predictions_gradcam.png", "Incorrect predictions")

## Interpretation Notes

After inspecting the saved figures, write observations here:

- Does the classifier focus on the pet face, body, fur texture, or background?
- Are wrong predictions caused by visually similar breeds?
- Are there cases where the heatmap highlights background instead of the animal?